In [1]:
import pandas as pd


In [2]:
dataset=pd.read_csv('final_processed_crash_dataset.csv')

dataset=dataset.loc[dataset['Vehicle'].isin(['E-bike','Bike','E-PMD'])]
dataset=dataset.loc[dataset['severity']!=-1]


/var/folders/y0/0nrj3m412p978185q3d2503sr02q24/T/ipykernel_14252/2039278705.py:1: DtypeWarning: Columns (11,17,51,52,54,56,57,58,60,61,66,67,68,69,70,71,73,74,76,77,78,84,86,87,88,99,100,101,102,106,110,113,114,115,116,117,118,119,120,121,122,124,133) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset=pd.read_csv('final_processed_crash_dataset.csv')


In [3]:
dataset['sev']=dataset['severity']

In [4]:
data_acc=dataset.drop_duplicates('Num_Acc')

var_for_acc = [
    'Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
    'int', 'atm', 'col', 'adr', 'lat', 'long', 'geometry', 'circ', 'nbv',
    'vosp', 'prof', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ',
    'Weekend', 'Year', 'Time category', 'Lighting conditions',
    'Weather conditions', 'Involved vehicle 1 type', 'Point of impact',
    'Reglementation', 'Cycle facilities', 'Agglomeration', 'Max speed',
    'Road type', 'Intersection', 'Crossroad',
    'number of involved vehicles', 'Long profile', 'Truck traffic',
    'Pavement', 'Road width', 'Surface condition', 'Time of day'
]

var_cont = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]


In [6]:
# ======================================
# One-hot encode selected categorical variables
# ======================================
var_added = pd.get_dummies(
    dataset[['Num_Acc', 'Cycle facilities', 'Pavement', 
             'Crossroad', 'positionnement_piste', 
             'Road type', 'Reglementation']]
)

# Convert boolean columns to integers
var_added = var_added.astype(int)

# ======================================
# Merge continuous variables from data_acc
# ======================================
var_added = var_added.merge(
    data_acc[['Num_Acc', 'surfacechaussee', 'pentemoyenne', 'largeurtrottoirdroit']],
    on='Num_Acc',
    how='left'
)

# ======================================
# Compute mean, min, and max for each column
# ======================================
results_list = [
    {
        'Column': col,
        'Mean': var_added[col].mean(),
        'Median': var_added[col].median(),
        'Min': var_added[col].min(),
        'Max': var_added[col].max()
    }
    for col in var_added.columns
]

# Convert the list of dictionaries into a DataFrame
results_df = pd.DataFrame(results_list)

# Display summary statistics
results_df.to_csv('summary_statistics_categorical_continuous.csv', index=False)


In [7]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal, shapiro, kstest

# ======================================
# Function to calculate Chi-square and p-value for each category
# Applies Cochran's rules for expected frequencies
# ======================================
def calculate_chi2_p_value_for_each_category(data, categorical_var):
    categories = data[categorical_var].unique()
    p_values = {}
    
    for category in categories:
        # Create a contingency table: severity vs this category
        contingency_table = pd.crosstab(data['severity'], data[categorical_var] == category)
        
        # Perform Chi-square test
        chi2, p, _, expected = chi2_contingency(contingency_table)
        
        # Apply Cochran's rules: expected counts must be >1 and no more than 20% <5
        if np.any(expected < 5):
            p_values[category] = np.nan  # Test not valid
        else:
            p_value_str = '<0.001' if p < 0.001 else str(round(p, 3))
            p_values[category] = p_value_str

    return p_values


# ======================================
# Function to count occurrences by severity for a categorical variable
# Returns counts, percentages, and formatted strings
# ======================================
def count_by_severity(data, categorical_var):
    # Count occurrences
    counts = data.groupby(['severity', categorical_var]).size().astype(int).reset_index(name='count')
    
    # Calculate percentage within the category
    total_counts = counts.groupby(categorical_var)['count'].transform('sum')
    counts['percentage'] = (counts['count'] / total_counts * 100).round(2)
    
    # Combine count and percentage into one string
    counts['count_percentage'] = counts.apply(
        lambda row: f"{row['count']} ({row['percentage']}%)", axis=1
    )
    
    return counts


# ======================================
# Function to compare continuous variables across severity groups
# Chooses ANOVA if normal, Kruskal-Wallis otherwise
# ======================================
def compare_continuous_variable(data, group_var, continuous_var):
    # Split data into groups
    groups_data = [
        data[data[group_var] == group][continuous_var].dropna().values 
        for group in data[group_var].unique()
    ]
    
    # Check normality using Kolmogorov-Smirnov test
    normal = all(
        kstest(group, 'norm', args=(group.mean(), group.std())).pvalue > 0.05
        for group in groups_data
    )

    if normal:
        # Parametric ANOVA test
        _, p_value = f_oneway(*groups_data)
    else:
        # Non-parametric Kruskal-Wallis test
        _, p_value = kruskal(*groups_data)
    
    # Format p-value
    if p_value < 0.001:
        return '<0.001'
    return str(round(p_value, 3))


# ======================================
# Define variables
# ======================================

# Categorical variables of interest
categorical_vars = [
    'Age category', 'Gender', 'Vehicle', 'User category', 'Helmet', 'sev', 'Point of impact_2',
    'Reflective jacket', 'Lighting conditions', 'Weather conditions', 
    'Point of impact', 'Maneuver', 'Cycle facilities', 'Accident location', 
    'Trip purpose', 'Max speed', 'Intersection', 'Crossroad', 
    'Long profile', 'Pavement', 'Surface condition', 'Road width',
    'vehicle_type_2', 'Maneuver_2', 'Gender_2', 'Age category involved', 
    'positionnement_piste', 'Road type'
]

# Continuous variables of interest
continuous_vars = [
    'age', 'age_2', 'Number of passengers',
    'number of involved vehicles', 'vma',
    'largeurchaussee', 'pentemoyenne', 'largeurtrottoirdroit'
]

# ======================================
# Generate pivot tables for categorical variables
# ======================================
pivot_results = []

for var in categorical_vars:
    # Decide which dataset to use
    df = data_acc if var in var_for_acc else dataset
    
    # Count occurrences and percentages
    counts_df = count_by_severity(df, var)
    
    # Pivot for table display
    pivot_df = counts_df.pivot(index=var, columns='severity', values='count_percentage').reset_index()
    pivot_df['Variable'] = var
    pivot_df['Category'] = pivot_df[var]
    pivot_df.drop(columns=var, inplace=True)
    
    # Compute p-values for each category
    p_values = calculate_chi2_p_value_for_each_category(df, var)
    pivot_df['p_value'] = pivot_df['Category'].map(p_values)
    
    pivot_results.append(pivot_df)


# ======================================
# Add continuous variables: median [Q1–Q3] and p-values
# ======================================
for var in continuous_vars:
    df = data_acc if var in var_for_acc else dataset
    df = df[['severity', var]].dropna(subset=[var])
    
    df = df[df[var] != 999]

    # Compute median and IQR
    stats_df = df.groupby('severity')[var].agg(
        median='median',
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    ).reset_index()
    
    # Format as "median [Q1–Q3]"
    stats_df['median_iqr'] = stats_df.apply(
        lambda row: f"{row['median']:.2f} [{row['q1']:.2f}–{row['q3']:.2f}]", axis=1
    )
    
    # Reformat for pivot display
    stats_df = stats_df[['severity', 'median_iqr']].set_index('severity').T
    stats_df.columns = [1, 2, 3]  # Assuming severity levels 1, 2, 3
    stats_df['Variable'] = var
    
    # Add p-value
    stats_df['p_value'] = compare_continuous_variable(df, 'severity', var)
    
    pivot_results.append(stats_df)


# ======================================
# Combine all results into one DataFrame
# ======================================
final_pivot_df = pd.concat(pivot_results, ignore_index=True)

# Reorder columns
cols_order = ['Variable', 'Category'] + [col for col in final_pivot_df.columns if col not in ['Variable', 'Category']]
final_pivot_df = final_pivot_df[cols_order]

# Rename variables for clarity
final_pivot_df = final_pivot_df.replace({
    'vehicle_type_2': 'Third-party vehicle type',
    'Maneuver_2': 'Third-party maneuver',
    'Gender_2': 'Third-party gender',
    'Point of impact_opposite':'Third-party impact location',
    'vma':'Speed limit',
    'surfacechaussee':'Road surface width',
    'pentemoyenne':'Average slope',
    'largeurtrottoirdroit':'Sidewalk width',
    'age':'Individual age',
    'number of involved vehicles':'Number of vehicles involved'
})


In [8]:
final_pivot_df.to_csv('descriptive_statistics_and_bivariate_tests.csv', index=False, sep=';')